---
title: "No Quantization for Small Model"
date: "2023-12-30"
categories: [nlp]
slug: "20231230-no-quantization-for-small-model"
description: "Quantization saves memory, but does it hurt quality on smaller 7B models? I ran the same summarization task across Q3, Q4, Q5 and Q8 variants. The differences were smaller than expected."
---



A while back I wrote a self-note / quick guide on [how to interpret the different quantization model codes](https://blog.mg3.xyz/20230910_llm_quantization_guide/), what they mean and how they affect performance, which I found useful as a general starting guide.

On this continued note I look at the same question but now specifically focused on how quantization affects performance of small 7B models, particularly [`OpenPipe/mistral-ft-optimized-1218`](https://huggingface.co/OpenPipe/mistral-ft-optimized-1218) or [`Weyaxi/Seraph-7B`](https://huggingface.co/Weyaxi/Seraph-7B) on summarization tasks.

![Battle of the Quants](/images/battle_quants.png)

*The Battle of the Quantized Models, 1927*


## Quantized Models & Summarization Task

For this study we will compare the performance of the following quantized model variants: `Q3_K_M`, `Q4_K_M`, `Q5_K_M` and `K8_0`. All of them are available for download on [HuggingFace](https://huggingface.co/TheBloke/mistral-ft-optimized-1218-GGUF), made available by [TheBloke](https://huggingface.co/TheBloke).

The task at hand will be the same as on the [last blog post](https://blog.mg3.xyz/20231216_una-cybertron-7b-v2_analysis/), which is *recursive summarization by parts* (i.e.: iteratively summarizing a long document into a small set of notes; see the blog post for more details or [this script](https://github.com/masta-g3/llmpedia/blob/main/workflow/d0_summarize.py) for the related code). I think this task is ideal for our purpose as the iterative nature of the process means that errors accumulate and amplify up to the final summary, making it easier to spot differences in performance. 

We will run the different model versions locally using LM Studio and evaluate them across three different axes:
- **Memory Usage**. How much memory does the model use during inference? This is the first metric where we expect to see a difference, as quantization is meant to reduce memory usage.
- **Inference Time**. How long does it take for the model to generate a summary? On the *recursive summarization by parts* framework we actually produce many mini-summaries, so we will look at the distribution of their creation runtime.
- **Summary Quality**. How good are the summaries, and how well do they adhere to the guidelines? GPT-4 will be our judge here, allowing us to expedite the process and remove any human bias. 

## Today's Reading: *Deja Vu - Contextual Sparsity for Efficient LLMs at Inference Time*
The document we will summarize is the *Deja Vu* LLM arxiv paper, which is a good choice as it is fairly technical but overall still digestible enough for the layman reader. For reference, here is a summary that the [GPT maestro produced](https://llmpedia.streamlit.app/?arxiv_code=2310.17157); we can use it as a benchmark for our own results:

> In their paper "Deja Vu: Contextual Sparsity for Efficient LLMs at Inference Time," the authors present a new method called "contextual sparsity" to improve the efficiency of Large Language Models (LLMs) during inference without compromising performance. Contextual sparsity identifies specific sets of attention heads and MLP parameters that produce similar outputs to dense models for certain inputs, accelerating LLM inference without affecting model quality. The authors introduce DEJAVU, a system that predicts contextual sparsity on the fly for each layer, reducing OPT-175B's inference latency by over 2x compared to existing methods. This approach has practical applications in reducing the computational cost of inference for large-scale LLMs, potentially enabling real-time inference on mobile devices or edge computing. DEJAVU uses contextual sparsity to improve LLM efficiency without sacrificing quality, reducing inference time by 4.5x compared to baseline models. The findings demonstrate that contextual sparsity can be applied to any pre-trained LLM without retraining and is compatible with various model architectures, offering significant efficiency gains without sacrificing accuracy.

## Results

Below are the final summaries produced by each quantized model, where we have requested a summary length of approximately 300 tokens.

<style>
/* Custom CSS to replicate Bulma styles for select box and text area */

/* Select box */
select.custom-select {
    display: block;
    width: 100%;
    height: calc(2.25em + 2px);
    padding: 0.375em 0.75em;
    font-size: 1em;
    line-height: 1.5;
    color: #495057;
    background-clip: padding-box;
    border: 1px solid #ced4da;
    border-radius: 0.25rem;
    transition: border-color 0.15s ease-in-out, box-shadow 0.15s ease-in-out;
}

/* Text area */
textarea.custom-textarea {
    display: block;
    width: 100%;
    height: calc(2.25em + 280px);
    padding: 0.375em 0.75em;
    font-size: 1em;
    line-height: 1.5;
    color: #495057;
    background-clip: padding-box;
    border: 1px solid #ced4da;
    border-radius: 0.25rem;
    transition: border-color 0.15s ease-in-out, box-shadow 0.15s ease-in-out;
    resize: vertical;
}
</style>

<!-- Select field for model selection with Bulma styles -->
<div class="field">
  <div class="control">
    <div class="select is-fullwidth">
      <select id="model_select" name="model_select" class="custom-select">
        <option value="Q3_K_M">Q3_K_M</option>
        <option value="Q4_K_M">Q4_K_M</option>
        <option value="Q5_K_M">Q5_K_M</option>
        <option value="K8_0">K8_0</option>
      </select>
    </div>
  </div>
</div>

<!-- Text area for displaying the summary with Bulma styles -->
<div class="field">
  <div class="control">
    <textarea id="summary_text" name="summary_text" class="custom-textarea"></textarea>
  </div>
</div>

<script>
// Define the dictionary with model names and summaries
var model_summaries = {
    'Q3_K_M': "The paper ‘Deja Vu: Contextual Sparsity for Efficient LLMs at Inference Time’ introduces a novel approach to improve Large Language Model (LLM) inference efficiency by exploiting contextual sparsity. This sparsity refers to the existence of small sets of attention heads and MLP parameters that yield approximately the same output as the dense model for a given input. Deja Vu (DV) can potentially lead to a 7x parameter reduction for specific inputs while maintaining accuracy, with an average of 85% structured sparse. The method uses a low-cost algorithm to predict contextual sparsity on the fly and an asynchronous and hardware-aware implementation that speeds up LLM inference without compromising quality or in-context learning ability. DV reduces the inference latency of OPT-175B by over 2× compared to the state-of-the-art FasterTransformer and over 6× compared to the widely used Hugging Face implementation, without sacrificing model quality. The method can potentially reduce the computational cost of LLMs while maintaining their impressive performance and in-context learning abilities. DV is compatible with quantization techniques and can be further optimized through hardware-aware sparse matrix multiply implementation. It achieves up to 30% FLOPs reduction in MLP blocks and 25% in attention blocks, with minimal latency increase. The method is applicable to various LLM architectures. Self-attention heads are crucial for learning different projection spaces to perform clustering, which can inform future research on LLM design. The paper does not present any unexpected findings or unusual results but emphasizes the practical benefits of its method. The main contributions of this work are more efficient use of memory resources, improved performance, and the potential for scalable LLM models that can be applied to a wide range of applications. No tables were presented in the paper with specific metrics or findings. The practical benefits of this research include more efficient use of memory resources, improved performance, and the potential for scalable LLM models that can be applied to a variety of applications.",
    'Q4_K_M': "‘Deja Vu: Contextual Sparsity for Efficient LLMs at Inference Time’ introduces a novel system that utilizes contextual sparsity to enhance the efficiency of large language models (LLMs) during inference time. This approach, called DEJAVU, employs a low-cost algorithm to predict contextual sparsity on the fly given inputs to each layer and an asynchronous and hardware-aware implementation that accelerates LLM inference. The proposed system significantly reduces the inference latency of OPT-175B by over 2× compared to the state-of-the-art FasterTransformer and over 6× compared to the widely used Hugging Face implementation, without compromising model quality. Contextual sparsity offers a promising approach for efficient LLM inference while maintaining performance and special abilities of pre-trained LLMs. DEJAVU is compatible with quantization techniques and can be applied to various LLM architectures and sizes. The paper provides a detailed analysis of the performance of DEJAVU on different models and inference settings, highlighting its potential for improving the efficiency of real-time natural language processing scenarios.",
    'Q5_K_M': "The paper introduces 'Deja Vu', a system that aims to reduce inference latency for large language models (LLMs) without compromising their quality or in-context learning ability. The concept of contextual sparsity is proposed, which involves small, input-dependent sets of attention heads and MLP parameters that yield approximately the same output as the dense model for a given input. DEJAVU is a system that uses a low-cost algorithm to predict contextual sparsity on the fly given inputs to each layer, along with an asynchronous and hardware-aware implementation that speeds up LLM inference. The paper validates that DEJAVU can reduce the inference latency of OPT-175B by over 2× compared to FasterTransformer and over 6× compared to Hugging Face implementation, without compromising model quality. Contextual sparsity does not require model retraining, preserves quality and in-context learning ability, and leads to speedup in wall-clock time on modern hardware. The paper addresses the challenge of speeding up inference-time sparse LLMs while maintaining quality and in-context learning abilities. Deja Vu is a method that aims to improve the efficiency and accessibility of large language models (LLMs) by reducing computational cost while maintaining accuracy. The approach focuses on contextual sparsity, which involves selectively discarding less important information from the model's internal representations during inference time. This is achieved through a two-stage process: first, a coarse-grained pruning step that removes a large portion of the model's parameters, and second, a fine-tuning step to recover accuracy lost during pruning. The authors propose a new metric called 'contextual sparsity ratio' (CSR) to measure the amount of information discarded by the method. Experiments show that Deja Vu can achieve up to 90% CSR without significant loss in accuracy, resulting in a 3x reduction in computational cost for inference. The authors also demonstrate that Deja Vu can be applied to different LLM architectures and pre-trained models, including GPT-2, BERT, and RoBERTa. Practical applications of Deja Vu include reducing the carbon footprint of LLMs by lowering energy consumption during inference, as well as making LLMs more accessible for resource-constrained devices and users. The method's ability to reduce computational cost without sacrificing accuracy could potentially lead to the development of more powerful and versatile LLMs in the future. Deja Vu is an innovative approach that addresses a critical issue in the field of LLM research, making it easier for researchers and developers to work with large language models while minimizing their environmental impact. The paper's findings could have significant implications for the development of more efficient and accessible AI systems in general.",
    'K8_0': "The 'Deja Vu' system, introduced in a recent paper, aims to reduce inference time for Large Language Models (LLMs) without compromising their quality or in-context learning ability. The authors propose contextual sparsity, which involves small sets of attention heads and MLP parameters that can approximate the output of a dense model for a given input. This approach has been shown to exist, be accurately predicted, and speed up LLM inference without sacrificing quality or in-context learning ability. A low-cost algorithm is proposed to predict contextual sparsity on the fly based on inputs to each layer. An asynchronous and hardware-aware implementation further speeds up LLM inference. The system was validated using OPT-175B, showing a reduction of 2× in inference latency compared to FasterTransformer and 6× compared to Hugging Face's implementation without compromising model quality. Contextual sparsity has practical applications in reducing the computational cost and energy consumption of LLMs during inference time without sacrificing performance. The paper also investigates contextual sparsity in Large Language Models, finding an average 85% structured sparse representation that can lead to a 7x parameter reduction while maintaining accuracy. The DEJAVU system is proposed to address the challenge of achieving efficiency through predicting and implementing contextual sparsity. Contextual sparsity has much better efficiency-accuracy trade-offs than non-contextual sparsity or static sparsity, with up to 7x improvement."
};

// Get the select field and text area elements
var model_select = document.getElementById('model_select');
var summary_text = document.getElementById('summary_text');

// Define a function that updates the summary text based on the selected model
function update_summary_text() {
    var model_name = model_select.value;
    summary_text.value = model_summaries[model_name];
}

// Attach the function to the select field
model_select.addEventListener('change', update_summary_text);

// Call the function once to initialize the text area
update_summary_text();
</script>

The first thing we notice is that not all models are able to produce a summary of the requested length, and only the `Q8_0` version is able to reach the desired ≤300 token mark. At some point, the other variants are unable to continue the summarization process, and from that iteration onwards they simply copy the text. We can see this behavior on the following token count plot: 

![Output summary token count by model type and iteration](/images/quantization_summary_token_count.png)

We also note that the `Q8_0` model also converges the fastest to the desired token length, while the more quantized variants seem to summarize the information at a slower rate. 

### Memory Usage and Runtime Speed

Before we continue looking at the summary quality, let's first look at the memory usage and runtime speed. The following table provides a summary view:

| Model | Memory Usage (GB) | Total Runtime (s) |
| --- |-------------------| --- |
| Q3_K_M | 8.74              | 891 |
| Q4_K_M | 9.54              | 675 |
| Q5_K_M | 10.30             | 859 |
| K8_0 | 12.80             | 564 |

For **memory usage**, we observe the expected behavior: total RAM utilized increases linearly with the number of bits used for quantization. `K8_0` uses almost 50% more memory than `Q3_K_M`, which is significant, but in absolute terms still a manageable amount for my M1 Mac (with 32GB of RAM).

Interestingly, from a **runtime** perspective, the `K0_8` model is the fastest to complete, but only because it converged to its final summary at a faster pace. In terms of *per chunk* runtime, we don't see clear differences, as indicated on the following graph:

![Distribution of chunk summarization runtime by model type](/images/quantization_runtime.png)

Performing a statistical tests over these distributions reveals that they do indeed have different means, although I believe this is driven more by the token count differences than by the quantization itself. The hypothesis makes sense considering there is no clear ordering or monotonicity in the results.

### Quality of Summaries

Finally we complete the assessment of the different models' summary quality. To do so, we will pass the four of them to GPT-4 leveraging the following prompt:


In [1]:
EVALUATOR_PROMPT = """
As a distinguished IT professor specializing in large language models and artificial intelligence, you have been approached by the Nobel Prize Association. They seek your expertise in evaluating four short summaries of a seminal AI paper. The author of the best summary will be awarded the Nobel Prize of the year. Recognizing the significance of this task, you approach it with utmost seriousness.

Below are the four summaries submitted by the candidates, identified as "candidates-Q". The  Nobel Prize Association has requested these summaries to be concise and informative, of 300 words or less.

Q3_K_M
{Q3_K_M}

Q4_K_M
{Q4_K_M}

Q5_K_M
{Q5_K_M}

K8_0
{K8_0}

Your assessment should focus on the following criteria:
0) Main Message: What is the core message of the review? What are its most interesting and practical aspects?
1) Conciseness: Did the author adhere to the 300 word limit, providing a succinct yet informative review?
2) Readability, Flow, and Prosaic Performance: How well-written is the review?  Is it consistent and clear? Does it offer deep analysis, or is it superficial?
3) Insights and Novelty: Does the review offer insightful, unexpected, or otherwise interesting findings? Does it provide precise metrics and figures to support its claims? Is it worth reading for an AI professional, or merely repetitive?
4) Overall Integrity: How commendable is the summary? Does it merit a Nobel Prize?

Please evaluate each summary in detail and then score each attribute on a scale of 1 to 5. The overall integrity will determine the final winner. Present your results in a numerical table. Compare and contrast to reach the best conclusion, and deliberate as needed. Review your decisions thoroughly before making a final commitment."""


The results are presented below, where we observe `K8_0` dominating by excelling in *conciseness* and *readability*:

| Model Version | Conciseness | Readability and Prosaic | Insights and Novelty | Overall Integrity |
|---------------|--------------|-------------|-------------------------|----------------------|
| **Q3_K_M**    | 2            | 4                       | 3                    | **3**             |
| **Q4_K_M**    | 5            | 5                       | 3                    | **4**             |
| **Q5_K_M**    | 2            | 4                       | 4                    | **3**             |
| **K8_0**      | 5            | 5                       | 4                    | **5**             |



## Conclusions
It seems there are not too many advantages of applying quantization to small models unless one is memory constrained. There seems to be a noticeable performance hit for the more heavily quantized models, and runtime might also degrade considering the model becomes less steerable and might not adhere to the user guidelines, including conciseness and token length. The technique seems better reserved for larger models, where the memory savings are more significant and the performance hit is likely less noticeable.